In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

## Environment Setup, Secrets & W&B Initialization

In [ ]:
import os
import random
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
import wandb
from kaggle_secrets import UserSecretsClient

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(seed=42)

user_secrets = UserSecretsClient()
wandb_api_key = user_secrets.get_secret("WanDB_API_KEY")
wandb.login(key=wandb_api_key)

wandb.init(
    project="DL-23f1000967-notebook-t22026",
    name="model1_tfidf_scratch_v3_hgb",
    config={
        "model_type": "FROM_scratch_HISTGB_LR_BLEND",
        "feature_extractor": "TFIDF_word+char+SVD",
        "word_tfidf_features": 4000,
        "char_tfidf_features": 3000,
        "svd_components": 150,
        "hgb_max_depth": 3,
        "hgb_learning_rate": 0.05,
        "hgb_max_iter": 200,
        "hgb_l2_regularization": 1.0,
        "lr_blend_weight": 0.2,
        "n_folds": 5
    }
)

config = wandb.config

## Load Competition Data

In [ ]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
OPTS = ['A', 'B', 'C', 'D', 'E']
print(f"Train Dataset Shape: {train_df.shape}")
print(f"Test Dataset Shape:  {test_df.shape}\n")

## Text Feature Engineering Engine (Reading from config)

In [ ]:
def build_corpus(df):
    corpus = df['prompt'].fillna("").tolist()
    for c in OPTS:
        corpus += df[c].fillna("").tolist()
    return corpus

full_corpus = build_corpus(train_df) + build_corpus(test_df)

word_vec = TfidfVectorizer(stop_words='english', max_features=config.word_tfidf_features,
                            sublinear_tf=True, ngram_range=(1, 2), min_df=2)
word_vec.fit(full_corpus)

char_vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5),
                            max_features=config.char_tfidf_features, sublinear_tf=True, min_df=2)
char_vec.fit(full_corpus)

svd = TruncatedSVD(n_components=config.svd_components, random_state=42)
svd.fit(word_vec.transform(full_corpus))

In [ ]:
def jaccard(a, b):
    sa, sb = set(a.lower().split()), set(b.lower().split())
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)

def build_features(df):
    n = len(df)
    feats = np.zeros((n, 5, 8), dtype=np.float32)

    prompts = df['prompt'].fillna("").tolist()
    opt_texts = {c: df[c].fillna("").tolist() for c in OPTS}

    p_word = word_vec.transform(prompts)
    p_char = char_vec.transform(prompts)
    p_svd = svd.transform(p_word)

    opt_word = {c: word_vec.transform(opt_texts[c]) for c in OPTS}
    opt_char = {c: char_vec.transform(opt_texts[c]) for c in OPTS}
    opt_svd = {c: svd.transform(opt_word[c]) for c in OPTS}

    for i in range(n):
        p_len = max(len(prompts[i]), 1)
        p_words = max(len(prompts[i].split()), 1)
        opt_lens = [len(opt_texts[c][i]) for c in OPTS]
        opt_word_counts = [len(opt_texts[c][i].split()) for c in OPTS]

        word_cos = np.array([cosine_similarity(p_word[i], opt_word[c][i])[0, 0] for c in OPTS])
        char_cos = [cosine_similarity(p_char[i], opt_char[c][i])[0, 0] for c in OPTS]
        svd_cos = [cosine_similarity(p_svd[i:i+1], opt_svd[c][i:i+1])[0, 0] for c in OPTS]
        jac = [jaccard(prompts[i], opt_texts[c][i]) for c in OPTS]
        len_ratio = [l / p_len for l in opt_lens]
        wc_ratio = [w / p_words for w in opt_word_counts]

        opt_vecs = [opt_word[c][i] for c in OPTS]
        centrality = [np.mean([cosine_similarity(opt_vecs[a], opt_vecs[b])[0, 0]
                                for b in range(5) if b != a]) for a in range(5)]

        word_cos_z = (word_cos - word_cos.mean()) / (word_cos.std() + 1e-6)

        for k in range(5):
            feats[i, k] = [word_cos[k], char_cos[k], svd_cos[k], jac[k],
                           len_ratio[k], wc_ratio[k], centrality[k], word_cos_z[k]]
    return feats

X_train_raw = build_features(train_df)
X_test_raw = build_features(test_df)

encoder = LabelEncoder()
y_train_raw = encoder.fit_transform(train_df['answer'])

print(f"Engineered Tensor (Train): {X_train_raw.shape}")
print(f"Engineered Tensor (Test):  {X_test_raw.shape}")

## Forward Neural Network Architecture

In [ ]:
def map_at_3(probs, y_true):
    top3 = np.argsort(-probs, axis=1)[:, :3]
    score = 0.0
    for i in range(len(y_true)):
        ranks = list(top3[i])
        if y_true[i] in ranks:
            score += 1.0 / (ranks.index(y_true[i]) + 1)
    return score / len(y_true)

## Training Loop with

In [ ]:
num_feats = X_train_raw.shape[2]
X_train_flat = X_train_raw.reshape(len(X_train_raw), -1)
X_test_flat = X_test_raw.reshape(len(X_test_raw), -1)

skf = StratifiedKFold(n_splits=config.n_folds, shuffle=True, random_state=42)

oof_probs = np.zeros((len(X_train_flat), 5))
test_probs_sum = np.zeros((len(X_test_flat), 5))
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train_flat, y_train_raw)):
    Xtr, Xva = X_train_flat[tr_idx], X_train_flat[va_idx]
    ytr, yva = y_train_raw[tr_idx], y_train_raw[va_idx]

    mean = Xtr.mean(0)
    std = Xtr.std(0) + 1e-6
    Xtr_n, Xva_n = (Xtr - mean) / std, (Xva - mean) / std
    Xte_n = (X_test_flat - mean) / std

    hgb = HistGradientBoostingClassifier(
        max_depth=config.hgb_max_depth, learning_rate=config.hgb_learning_rate,
        max_iter=config.hgb_max_iter, l2_regularization=config.hgb_l2_regularization,
        random_state=42)
    hgb.fit(Xtr_n, ytr)

    lr = LogisticRegression(max_iter=2000)
    lr.fit(Xtr_n, ytr)

    w = config.lr_blend_weight
    va_probs = (1 - w) * hgb.predict_proba(Xva_n) + w * lr.predict_proba(Xva_n)
    te_probs = (1 - w) * hgb.predict_proba(Xte_n) + w * lr.predict_proba(Xte_n)

    fold_map3 = map_at_3(va_probs, yva)
    wandb.log({"fold": fold, "val_map3": fold_map3})

    oof_probs[va_idx] = va_probs
    test_probs_sum += te_probs
    fold_scores.append(fold_map3)
    print(f"Fold {fold} | val MAP@3: {fold_map3:.4f}")

oof_map3 = map_at_3(oof_probs, y_train_raw)
wandb.log({"mean_fold_map3": np.mean(fold_scores), "oof_map3": oof_map3})
wandb.finish()
print(f"Mean fold MAP@3: {np.mean(fold_scores):.4f}")
print(f"Overall OOF MAP@3: {oof_map3:.4f}")

## Rank Inference & Submission Export

In [ ]:
test_probs = test_probs_sum / config.n_folds
classes = np.array(OPTS)
map_predictions = []

for row_probs in test_probs:
    top_3_indices = np.argsort(row_probs)[::-1][:3]
    top_3_labels = classes[top_3_indices]
    map_predictions.append(" ".join(top_3_labels))

submission_df = pd.DataFrame({
    'id': test_df['id'],
    'prediction': map_predictions
})
submission_df.to_csv('submission.csv', index=False)
display(submission_df.head(5))